# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the geolocation dataset as part of the Data Quality Monitoring System for e-commerce operations. It performs data inspection, profiling, cleaning, and validation to identify and address data quality issues before the dataset is used for downstream analysis.

This script focuses on the preparation of the Olist orders dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports order lifecycle analysis by tracking each order from purchase through approval, shipping, and delivery, forming the foundation for analyzing customer purchasing behavior, delivery performance, and operational efficiency in SQL and Power BI.

### Import The Libraries

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### Load The Dataset

In [22]:
orders_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_orders_dataset.csv")

## **ORDERS DATASET**

### 1. Data Inspection

In [23]:
# The first first five rows of the orders dataset
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [24]:
# The number of rows and columns in the orders dataset
orders_df.shape

(99441, 8)

In [25]:
# Check the columns names and data types in the orders dataset
orders_df.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

The dataset contains eight columns, all of which are currently stored as the `object` data type. While the identifier and status fields are correctly represented as text, the five date-related columns (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, and `order_estimated_delivery_date`) should be converted to the `datetime` data type. This conversion is necessary to enable accurate date calculations, chronological validation, and time-based analysis throughout the project.


In [26]:
# Check for missing values in the orders dataset
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

The dataset contains no missing values in the `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp`, and `order_estimated_delivery_date` columns. However, missing values are present in `order_approved_at` (160), `order_delivered_carrier_date` (1,783), and `order_delivered_customer_date` (2,965). These missing values are expected in certain cases, such as orders that were cancelled, unavailable, or had not progressed to later stages of the order lifecycle. Their validity will be assessed alongside the corresponding order status during the data profiling process.


In [27]:
# Check for duplicate rows in the orders dataset
for column in orders_df.columns:
    duplicates = orders_df[column].duplicated().sum()
    print(f"{column}: {duplicates}")

order_id: 0
customer_id: 0
order_status: 99433
order_purchase_timestamp: 566
order_approved_at: 8707
order_delivered_carrier_date: 18422
order_delivered_customer_date: 3776
order_estimated_delivery_date: 98982


The `order_id` column contains no duplicate values, confirming that each record represents a unique customer order. Duplicate values observed in the remaining columns are expected because multiple orders can share the same customer, status, purchase date, or estimated delivery date. Therefore, no duplicate-related data quality issues were identified during the initial inspection.


In [28]:
# Check the summary statistics of the orders dataset
orders_df.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


The summary statistics indicate that the dataset contains **99,441 order records** and **8 variables** describing the order lifecycle. Both the `order_id` and `customer_id` columns contain 99,441 unique values, confirming that each record represents a unique customer order. The `order_status` column contains eight distinct categories, with **"delivered"** being the most frequent status (96,478 records), indicating that the majority of orders were successfully completed. The summary statistics also reveal fewer non-null values in the `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` columns, suggesting the presence of missing timestamps that require further investigation during the data profiling stage.


### 2. Data Profiling

#### Order Status and Lifecycle Analysis

In [29]:
# Count the number of occurrences of each unique value in the "order_status" column
orders_df["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [30]:
orders_df.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda x: x.isnull().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


The investigation confirms that the majority of missing timestamp values are consistent with the order lifecycle. Orders with statuses such as **created, processing, invoiced, cancelled, shipped, and unavailable** contain missing timestamps that accurately reflect their progression through the fulfilment process and therefore do not represent data quality issues.
However, a small number of **delivered orders** contain missing approval, carrier dispatch, or customer delivery timestamps. Since delivered orders are expected to have completed the full order lifecycle, these records will be examined further during the chronological validation stage to determine whether they represent genuine data quality inconsistencies.

#### Profiling Missing Values

In [31]:
# Calculate the percentage of missing values
missing_percentage = (
    orders_df.isnull().sum() / len(orders_df) * 100
).round(2)

missing_percentage

order_id                         0.00
customer_id                      0.00
order_status                     0.00
order_purchase_timestamp         0.00
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
order_estimated_delivery_date    0.00
dtype: float64

In [36]:
orders_df[
    orders_df["order_delivered_customer_date"].isnull()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

The investigation of missing **`order_delivered_customer_date`** values confirms that the majority of missing timestamps are associated with order statuses where customer delivery has not yet occurred. Most missing values belong to **shipped**, **cancelled**, **unavailable**, **invoiced**, **processing**, and **created** orders, indicating that these null values are consistent with the expected order lifecycle rather than data quality issues.

Only a small number of records with the **delivered** (8) and **approved** (2) statuses contain missing customer delivery timestamps. Since these orders are expected to progress further in the fulfilment process, they will be retained for chronological validation and considered during the data cleaning phase.

#### Unique Values

In [32]:
# Count unique values in each column
orders_df.nunique()

order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90733
order_delivered_carrier_date     81018
order_delivered_customer_date    95664
order_estimated_delivery_date      459
dtype: int64

#### Date Ranges

In [33]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_df[date_columns] = orders_df[date_columns].apply(pd.to_datetime)

orders_df[date_columns].agg(["min", "max"])

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


#### Customer Activity

In [34]:
orders_df["customer_id"].value_counts().describe()

count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: count, dtype: float64

#### Order Lifecycle Completeness

In [35]:
orders_df[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ]
].notnull().sum()

order_approved_at                99281
order_delivered_carrier_date     97658
order_delivered_customer_date    96476
dtype: int64

In [38]:
# Analyse lifecycle completeness by order status
orders_df.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].agg(lambda x: x.notnull().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,2,0,0
canceled,484,75,6
created,0,0,0
delivered,96464,96476,96470
invoiced,314,0,0
processing,301,0,0
shipped,1107,1107,0
unavailable,609,0,0


The order lifecycle completeness analysis indicates that the progression of orders is largely consistent with the expected fulfilment process. **Created** orders contain no lifecycle timestamps beyond order creation, while **approved**, **processing**, **invoiced**, and **shipped** orders progressively accumulate approval, carrier dispatch, and customer delivery timestamps as expected. The vast majority of **delivered** orders contain all three lifecycle timestamps, demonstrating a complete fulfilment process.

A small number of **delivered** orders remain incomplete due to missing approval, carrier dispatch, or customer delivery timestamps. These records will be investigated further during chronological validation to determine whether they represent genuine data quality issues. Additionally, some **cancelled** orders contain approval, carrier dispatch, or delivery timestamps, reflecting that order cancellations can occur at different stages of the fulfilment process rather than indicating inconsistent data.

In [39]:
# Orders approved before being purchased
approved_before_purchase = orders_df[
    orders_df["order_approved_at"] < orders_df["order_purchase_timestamp"]
]

print(f"Records found: {len(approved_before_purchase)}")

approved_before_purchase.head()

Records found: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


The chronological investigation found **no instances** where an order was approved before it was purchased. This confirms that the relationship between the **`order_purchase_timestamp`** and **`order_approved_at`** columns is consistent with the expected business process. No data quality issues were identified for this validation rule.

In [43]:
# Carrier dispatch before order approval
carrier_before_approval = orders_df[
    orders_df["order_delivered_carrier_date"] < orders_df["order_approved_at"]
]

print(f"Records found: {len(carrier_before_approval)}")

Records found: 1359


In [52]:
# Calculate the time difference between carrier dispatch and order approval
carrier_before_approval = carrier_before_approval.copy()

carrier_before_approval["approval_delay"] = (
    carrier_before_approval["order_approved_at"]
    - carrier_before_approval["order_delivered_carrier_date"]
)
carrier_before_approval["approval_delay"].describe()

count                         1359
mean     1 days 00:45:07.153053715
std      4 days 19:18:28.055754668
min                0 days 00:00:21
25%         0 days 01:24:55.500000
50%                0 days 17:10:04
75%                1 days 01:57:24
max              171 days 05:15:22
Name: approval_delay, dtype: object

In [48]:
carrier_before_approval[
    [
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "approval_delay",
    ]
].head(10)

,order_id,order_status,order_approved_at,order_delivered_carrier_date,approval_delay
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00,1 days 08:37:02
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14,0 days 23:06:08
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00,2 days 10:34:53
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00,0 days 00:07:53
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00,3 days 09:28:09
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-05 16:33:06,2018-07-05 14:50:00,0 days 01:43:06
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-29 23:30:52,2018-07-26 14:46:00,3 days 08:44:52
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 16:52:39,2018-05-07 15:09:00,0 days 01:43:39
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 14:05:13,2018-08-16 13:27:00,0 days 00:38:13
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-05 16:31:26,2018-07-04 12:14:00,1 days 04:17:26


The chronological investigation identified **1,359** records where the **`order_delivered_carrier_date`** precedes the **`order_approved_at`** timestamp. Further analysis revealed that the time differences range from **21 seconds** to **171 days**, with a median delay of approximately **17 hours** and an average delay of approximately **one day**.

These findings suggest that the records do not represent a single type of anomaly. Smaller time differences may be attributable to system processing delays or timestamp precision, whereas larger delays could indicate delayed approval recording or potential data quality issues. The distribution of these delays will be examined further before determining whether any cleaning actions are necessary.

In [49]:
# Categorise approval delays
carrier_before_approval["approval_delay"].dt.total_seconds().describe()

count    1.359000e+03
mean     8.910715e+04
std      4.151081e+05
min      2.100000e+01
25%      5.095500e+03
50%      6.180400e+04
75%      9.344400e+04
max      1.479332e+07
Name: approval_delay, dtype: float64

In [50]:
# Categorise approval delays
delay_hours = (
    carrier_before_approval["approval_delay"]
    .dt.total_seconds() / 3600
)

print(f"Less than 1 hour: {(delay_hours < 1).sum()}")
print(f"1 hour to 24 hours: {((delay_hours >= 1) & (delay_hours < 24)).sum()}")
print(f"More than 24 hours: {(delay_hours >= 24).sum()}")

Less than 1 hour: 242
1 hour to 24 hours: 659
More than 24 hours: 458


The chronological investigation identified **1,359** records where the **`order_delivered_carrier_date`** precedes the **`order_approved_at`** timestamp. Further analysis revealed that these differences vary considerably in magnitude. Approximately **17.81%** of the affected records have delays of less than one hour, **48.49%** fall between one and twenty-four hours, and **33.70%** exceed twenty-four hours.

The variation in delay durations suggests that these records do not represent a single, consistent data quality issue. Smaller differences may reflect system processing delays or timestamp precision, whereas larger differences could be attributed to delayed approval recording, business exceptions, or system synchronization issues. As no authoritative source is available to determine which timestamp is correct, these records will be retained without modification during the data cleaning phase.

In [42]:
# Customer delivery before carrier dispatch
delivered_before_carrier = orders_df[
    orders_df["order_delivered_customer_date"] < orders_df["order_delivered_carrier_date"]
]

print(f"Records found: {len(delivered_before_carrier)}")

Records found: 23


In [53]:
# Create a copy to avoid SettingWithCopyWarning
delivered_before_carrier = delivered_before_carrier.copy()

# Calculate the time difference
delivered_before_carrier["delivery_delay"] = (
    delivered_before_carrier["order_delivered_carrier_date"]
    - delivered_before_carrier["order_delivered_customer_date"]
)

# Summary statistics
delivered_before_carrier["delivery_delay"].describe()

count                           23
mean     3 days 06:27:18.478260869
std      3 days 17:18:29.050924326
min                0 days 00:23:18
25%                0 days 23:20:18
50%                1 days 15:51:52
75%         5 days 08:45:32.500000
max               16 days 02:18:29
Name: delivery_delay, dtype: object

In [54]:
delivered_before_carrier[
    [
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "delivery_delay"
    ]
]

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,delivery_delay
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56,2 days 21:25:02
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51,1 days 02:54:50
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28,1 days 05:08:34
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,6 days 00:14:20
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41,0 days 04:11:01
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46,0 days 17:50:07
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35,0 days 23:06:19
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38,0 days 00:23:18
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56,1 days 00:27:51
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01,7 days 21:05:42


The chronological investigation identified **23** records where the **`order_delivered_customer_date`** occurs before the **`order_delivered_carrier_date`**. Further analysis showed that the time differences range from **23 minutes** to **16 days**, with a median delay of approximately **1 day and 16 hours**.

Unlike minor timestamp discrepancies that may arise from system processing delays, the majority of these records exhibit differences measured in several days. These findings suggest the presence of chronological inconsistencies within a small subset of the dataset. However, as there is no authoritative source to determine which timestamp is incorrect, these records will be retained without modification and documented as potential data quality anomalies.

In [44]:
# Estimated delivery before purchase
estimated_before_purchase = orders_df[
    orders_df["order_estimated_delivery_date"] < orders_df["order_purchase_timestamp"]
]

print(f"Records found: {len(estimated_before_purchase)}")


Records found: 0


The chronological investigation found **no records** where the **`order_estimated_delivery_date`** occurs before the **`order_purchase_timestamp`**. This confirms that the estimated delivery dates are chronologically consistent with the order purchase dates. No data quality issues were identified for this business rule, and no cleaning actions are required.

### 3. Data Cleaning

#### Convert Timestamp Columns

In [55]:
# Convert timestamp columns to datetime format
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_df[date_columns] = orders_df[date_columns].apply(
    pd.to_datetime,
    errors="coerce"
)

In [56]:
# Validate the updated data types
orders_df[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

The five timestamp columns were successfully converted from the **`object`** data type to **`datetime64[ns]`**, enabling accurate chronological analysis, date arithmetic, and time-based reporting. The conversion preserves valid timestamp values while ensuring that any invalid date formats would be safely converted to **`NaT`** without interrupting the data preparation process.

#### Additional Decisions

The missing timestamp values were intentionally retained because the profiling phase confirmed that they are consistent with the expected order lifecycle. Orders with statuses such as **created**, **processing**, **invoiced**, **shipped**, **cancelled**, and **unavailable** naturally lack one or more lifecycle timestamps, reflecting valid business scenarios rather than incomplete or erroneous data.
As these null values accurately represent the operational state of each order, no imputation or record removal was performed. Retaining the original values preserves the integrity of the dataset and prevents the introduction of artificial or misleading information.

The chronological anomalies identified during profiling were retained without modification. Although a small number of records exhibited inconsistencies in the sequence of lifecycle timestamps, further investigation found insufficient evidence to determine which timestamp values were incorrect.
In the absence of an authoritative source, no automated corrections were applied. Preserving the original values maintains the integrity of the source data and ensures that any identified anomalies remain transparent and traceable for future analysis.

In [59]:
# Final Validation: Check for any remaining data quality issues

# Dataset dimensions
display(orders_df.shape)

# Data types
display(orders_df.dtypes)

# Missing values
display(orders_df.isnull().sum())

# Duplicate records
display(orders_df.duplicated().sum())

# Chronological integrity checks
purchase_before_approval = len(
    orders_df[
        orders_df["order_approved_at"] <
        orders_df["order_purchase_timestamp"]
    ]
)

carrier_before_approval = len(
    orders_df[
        orders_df["order_delivered_carrier_date"] <
        orders_df["order_approved_at"]
    ]
)

delivery_before_carrier = len(
    orders_df[
        orders_df["order_delivered_customer_date"] <
        orders_df["order_delivered_carrier_date"]
    ]
)

estimated_before_purchase = len(
    orders_df[
        orders_df["order_estimated_delivery_date"] <
        orders_df["order_purchase_timestamp"]
    ]
)

print(f"Purchase before Approval violations: {purchase_before_approval}")
print(f"Carrier before Approval violations: {carrier_before_approval}")
print(f"Delivery before Carrier violations: {delivery_before_carrier}")
print(f"Estimated Delivery before Purchase violations: {estimated_before_purchase}")

# Final dataset information
orders_df.info()

(99441, 8)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

np.int64(0)

Purchase before Approval violations: 0
Carrier before Approval violations: 1359
Delivery before Carrier violations: 23
Estimated Delivery before Purchase violations: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1

### 4. Export The Cleaned Dataset

In [60]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [62]:
orders_df.to_csv(
    CLEANED_DATA_PATH / "olist_orders_cleaned.csv",
    index=False
)